# P5 — Sequence Modeling with RNNs

**Goal:** Apply basic and advanced **RNN / LSTM / GRU, Encoder-Decoder** architectures to a sequence-based sentiment classification task.

---

## Dataset (Required)

### IMDB Movie Reviews (Binary Sentiment Classification)
**Access method (required):**
```python
from tensorflow.keras.datasets import imdb
(x_train, y_train), (x_test, y_test) = ...
```

You must create a validation split from training data and pad/truncate sequences to a fixed length.

---
## What you will implement

You will implement and compare the following **sequence models**:

- **Vanilla RNN**
- **LSTM**
- **GRU**
- **Stacked LSTM + Dropout**
- **Encoder–Decoder LSTM**
- **LSTM + GRU Hybrid**

---

## Q0 — Setup (Ungraded)
#### Import libraries, set seeds, and verify TensorFlow / TFDS.

In [3]:
# ============================================================
# Q0) Environment Setup
# ============================================================

import os
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


TensorFlow: 2.21.0


---
## ✅ Student Instructions (Start Here)

Your work begins in the **next code cells (Q1–Q11)** and continues with the **Markdown responses (Q12–Q15)**.  
These correspond to the questions listed in the assignment description on **Canvas**. Follow the instructions provided in the **preceding Markdown cells** for each step.

### Tasks

This assignment focuses on **sequence modeling for text classification** using recurrent neural networks.

You will:

- Train and evaluate the following **sequence models**:
  - **Vanilla RNN**
  - **LSTM**
  - **GRU**

- Implement additional **advanced architectures**:
  - **Stacked LSTM + Dropout**
  - **Encoder–Decoder LSTM**
  - **LSTM + GRU Hybrid**

- Use the **IMDB Movie Reviews dataset** for **binary sentiment classification**.

- Perform a **comparative analysis** of the models, including:
  - training convergence behavior
  - validation and test performance
  - explanation of architectural differences across models

Ensure that all models are **computationally feasible** to train on **CPU-only environments** by using the recommended hyperparameters unless you have access to a GPU (e.g. Google Colab).

---

## Q1 — Load Dataset & Inspect

Use the **IMDB Movie Reviews dataset** from Keras and inspect its basic structure.

### Student Tasks

- Load the IMDB dataset using `tensorflow.keras.datasets.imdb` with a vocabulary size of **10,000 words** by frequency.

- Split the training data into **training** and **validation** sets.

- Inspect the dataset by:
  - Printing the **number of training and test samples**
  - Displaying the **label distribution (train)**
  - Printing one example **Sequence length (train)**

---

In [4]:
# ============================================================
# Question Q1 — Load IMDB Dataset (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Load the IMDB Movie Reviews dataset
# 2) Print the number of training and test examples
# 3) Check the label distribution in the training set
# 4) Inspect sequence length statistics
# ============================================================

import numpy as np
#from tensorflow.keras.datasets import imdb
from keras.datasets import imdb
# TODO 1: Define vocabulary size
VOCAB_SIZE = 10000

# TODO 2: Load the IMDB dataset
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# TODO 3: Print dataset sizes
print("Train examples:", len(x_train))
print("Test examples: ", len(x_test))

# TODO 4: Print label distribution
print("Label distribution (train):", np.bincount(y_train))

# TODO 5: Compute sequence length statistics
train_lengths = np.array([len(s) for s in x_train])

print(
    "Sequence length (train): min/median/max =",
    train_lengths.min(),
    int(np.median(train_lengths)),
    train_lengths.max()
)


Train examples: 25000
Test examples:  25000
Label distribution (train): [12500 12500]
Sequence length (train): min/median/max = 11 178 2494


---

## Q2 — Validation Split & Sequence Padding

Prepare the dataset for training by creating a **validation split** and converting all sequences to a **fixed length**.

### Student Tasks

- Create a **validation set** from the training data (`VAL_SIZE = 5000`).  
  Use a **deterministic split from the end of the training set**.

- Define a maximum sequence length **`MAX_LEN`** (e.g., 200–300 tokens) based on the sequence statistics observed in **Q1**.

- Apply **sequence padding and truncation** using `pad_sequences` so that all reviews have the same length:
  - Use **post-padding** 
  - Use **post-truncation** 

- Generate padded datasets for:
  - `x_train_pad`
  - `x_val_pad`
  - `x_test_pad`

- Keep it **consistent across all models**.

- Print the shapes of the padded arrays to confirm the preprocessing step completed successfully.

---

In [5]:

# ============================================================
# Question Q2 — Validation Split & Sequence Padding (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define MAX_LEN and validation size
# 2) Create a validation split from the training data
# 3) Pad/truncate sequences to a fixed length
# 4) Verify resulting dataset shapes
# ============================================================

from tensorflow.keras.preprocessing.sequence import pad_sequences

# TODO 1: Define sequence length and validation size
MAX_LEN = 300
VAL_SIZE = 500

# TODO 2: Create validation split from the end of the training set
x_val, y_val = x_train[-VAL_SIZE:], y_train[-VAL_SIZE:]
x_train2, y_train2 = x_train[:-VAL_SIZE], y_train[:-VAL_SIZE]


# TODO 3: Apply padding and truncation
x_train_pad = pad_sequences(
    x_train2,
    maxlen=MAX_LEN,
    padding="pre",  # first time through, i put "same" thinking it was the Conv style padding
    truncating="pre", #first time through i put "True" thinking it would enable it, turns out only correct values are
    # pre or post per the tensorflow documentation
)

x_val_pad = pad_sequences(
    x_val,
    maxlen=MAX_LEN,
    padding="pre",
    truncating="pre",
)

x_test_pad = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="pre",
    truncating="pre",
)

# Print dataset shapes
print("x_train_pad:", x_train_pad.shape)
print("x_val_pad:  ", x_val_pad.shape)
print("x_test_pad: ", x_test_pad.shape)

x_train_pad: (24500, 300)
x_val_pad:   (500, 300)
x_test_pad:  (25000, 300)


---

## Q3 — Build `tf.data` Pipelines

Create efficient **data pipelines** for training, validation, and testing using **TensorFlow `tf.data`**.

### Student Tasks

- Convert the padded datasets into **TensorFlow datasets** using `tf.data.Dataset.from_tensor_slices`.

- Create datasets for:
  - **training**
  - **validation**
  - **testing**

- Apply the following pipeline steps:
  - **shuffle** the training dataset
  - **batch** the datasets using an appropriate batch size
  - use **prefetching** to improve training performance

- Ensure the pipelines are ready to be used directly in **model training with `model.fit()`**.


--- 

In [6]:
# ============================================================
# Question Q3 — tf.data Pipelines (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define batch size and AUTOTUNE
# 2) Create TensorFlow datasets from the padded arrays
# 3) Apply shuffle, batch, and prefetch operations
# 4) Prepare pipelines for training, validation, and testing
# ============================================================

# TODO 1: Define batch size and autotune
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# TODO 2: Create training dataset
train_ds = tf.data.Dataset.from_tensor_slices((x_train_pad, y_train2))

# TODO 3: Apply shuffle, batch, and prefetch
train_ds = train_ds.shuffle(1000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 4: Create validation dataset
val_ds = tf.data.Dataset.from_tensor_slices((x_val_pad, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 5: Create test dataset
test_ds = tf.data.Dataset.from_tensor_slices((x_test_pad, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Pipelines ready.")
print("x_train_pad =", x_train_pad)
print("val_ds =", val_ds)
print("val_ds =", test_ds)

Pipelines ready.
x_train_pad = [[   0    0    0 ...   19  178   32]
 [   0    0    0 ...   16  145   95]
 [   0    0    0 ...    7  129  113]
 ...
 [   0    0    0 ...   18   72 4366]
 [   0    0    0 ...   79   25 1487]
 [   6 3539  292 ...    6 1281 1694]]
val_ds = <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 300), dtype=tf.int32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
val_ds = <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 300), dtype=tf.int32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


---

## Q4 — Define Training Utilities

In this step, you will create **reusable helper functions** that simplify model training, evaluation, and experiment management.

These utilities will be used for **all models later in the assignment**.

### Student Tasks

1. **Create training callbacks**

   Define a function that returns commonly used training callbacks, including:

   - **EarlyStopping** to stop training when validation performance stops improving.
   - **ReduceLROnPlateau** to automatically reduce the learning rate when validation loss plateaus.
   - **ModelCheckpoint** to save the **best-performing model** during training.

2. **Define a model compilation function**

   Implement a function that compiles a model using:

   - **Adam optimizer**
   - **Binary cross-entropy loss** for sentiment classification
   - **Accuracy** as the evaluation metric

3. **Create a training function**

   Implement a function that trains a model using:

   - the **training dataset**
   - the **validation dataset**
   - the callbacks defined above

4. **Create an evaluation function**

   Implement a function that evaluates a trained model on a dataset and reports:

   - **loss**
   - **accuracy**

These functions will help keep the notebook **organized, reusable, and consistent across experiments**.

---

In [8]:
# ============================================================
# Question Q4 — Training Utilities (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define callbacks for training
# 2) Compile the model with optimizer, loss, and metrics
# 3) Train the model using training and validation datasets
# 4) Evaluate the trained model on a dataset
# ============================================================
print(type(y_train2), len(y_train2))
print(set(y_train2))
# TODO 1: Define training callbacks
def build_callbacks(run_name: str):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
        tf.keras.callbacks.ModelCheckpoint(
           filepath=rf"C:\Users\jevin\Documents\MichiganTechGradSchool\MachineLearning\{run_name}.keras",
            monitor="val_accuracy",
            save_best_only=True,
        ),
    ]


# TODO 2: Compile model
def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),

        metrics=["accuracy"],
    )
    return model


# TODO 3: Train model
def train_model(model, run_name: str, epochs=8):
    history = model.fit(
        x_train_pad,
        y_train2, # I had to this for it to work, looking at other our other hw assignements i think i need it,
                  # otherwise it would have output data to train
        validation_data=val_ds,
        epochs=epochs,
        callbacks=build_callbacks(run_name),
        verbose=1,
    )
    return history


# TODO 4: Evaluate model
def evaluate_model(model, name: str, ds):
    loss, acc = model.evaluate(ds, verbose=0)
    print(f"{name}: loss={loss:.4f}, acc={acc:.4f}")
    return loss, acc


<class 'numpy.ndarray'> 24500
{np.int64(0), np.int64(1)}


---

## Q5 — Model A: Vanilla RNN

Build a **Vanilla RNN** model for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add a **SimpleRNN layer** for sequence processing.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.

---

In [9]:
# ============================================================
# Question Q5 — Model A: Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define embedding dimension and RNN units
# 2) Build a Sequential RNN model
# 3) Add Input, Embedding, RNN, and Dense layers
# 4) Compile the model
# 5) Display the model summary
# ============================================================

from tensorflow.keras import layers

# TODO 1: Define model hyperparameters (e.g., 128, 128)
EMBED_DIM = 128
RNN_UNITS = 128

# TODO 2: Build the Sequential model
rnn_model = tf.keras.Sequential([
    
    # TODO 3: Input layer
    layers.Input(shape=(MAX_LEN,)),
    
    # TODO 4: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
    
    # TODO 5: Simple RNN layer
    layers.SimpleRNN(RNN_UNITS),
    
    # TODO 6: Output layer
    layers.Dense(1, activation="sigmoid"),
    
], name="vanilla_rnn")

# TODO 7: Compile the model
rnn_model = compile_model(rnn_model, lr=0.001)

# Print model summary
rnn_model.summary()

# ============================================================
# Train and Evaluate Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the Vanilla RNN model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the RNN model
# TODO 1: Train the RNN model
history_rnn = train_model(rnn_model, run_name="proj5_vanilla_rnn", epochs=8)


# TODO 2: Evaluate on validation set
evaluate_model(rnn_model, "Validation (RNN)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(rnn_model, "Test (RNN)", test_ds)

Model: "vanilla_rnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 33s 41ms/step - accuracy: 0.6062 - loss: 0.6425 - val_accuracy: 0.5900 - val_loss: 0.6511 - learning_rate: 0.0010
Epoch 2/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 31s 41ms/step - accuracy: 0.7060 - loss: 0.5645 - val_accuracy: 0.7300 - val_loss: 0.5685 - learning_rate: 0.0010
Epoch 3/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 33s 43ms/step - accuracy: 0.7797 - loss: 0.4751 - val_accuracy: 0.7260 - val_loss: 0.5949 - learning_rate: 0.0010
Epoch 4/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 36s 47ms/step - accuracy: 0.8337 - loss: 0.3845 - val_accuracy: 0.7380 - val_loss: 0.6232 - learning_rate: 5.0000e-04
Epoch 5/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 33s 43ms/step - accuracy: 0.8573 - loss: 0.3440 - val_accuracy: 0.7360 - val_loss: 0.5975 - learning_rate: 2.5000e-04
Epoch 6/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 34s 45ms/step - accuracy: 0.8845 - loss: 0.2912 - val_accuracy: 0.7440 - val_loss: 0.5944 - learning_rate: 1.2500e-04
Epoch 7/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 35s 46ms/step - accuracy: 0.898

(0.5143851041793823, 0.793720006942749)

In [10]:
# ============================================================
# Train and Evaluate Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the Vanilla RNN model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the RNN model
# TODO 1: Train the RNN model
history_rnn = train_model(rnn_model, run_name="proj5_vanilla_rnn", epochs=8)


# TODO 2: Evaluate on validation set
evaluate_model(rnn_model, "Validation (RNN)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(rnn_model, "Test (RNN)", test_ds)

Epoch 1/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 33s 42ms/step - accuracy: 0.9082 - loss: 0.2442 - val_accuracy: 0.7540 - val_loss: 0.5778 - learning_rate: 1.5625e-05
Epoch 2/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 33s 43ms/step - accuracy: 0.9095 - loss: 0.2409 - val_accuracy: 0.7600 - val_loss: 0.5749 - learning_rate: 1.5625e-05
Epoch 3/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 34s 45ms/step - accuracy: 0.9114 - loss: 0.2380 - val_accuracy: 0.7580 - val_loss: 0.5758 - learning_rate: 1.5625e-05
Epoch 4/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 32s 42ms/step - accuracy: 0.9118 - loss: 0.2357 - val_accuracy: 0.7620 - val_loss: 0.5719 - learning_rate: 7.8125e-06
Epoch 5/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 34s 45ms/step - accuracy: 0.9130 - loss: 0.2335 - val_accuracy: 0.7640 - val_loss: 0.5716 - learning_rate: 7.8125e-06
Epoch 6/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 37s 48ms/step - accuracy: 0.9140 - loss: 0.2321 - val_accuracy: 0.7680 - val_loss: 0.5701 - learning_rate: 7.8125e-06
Epoch 7/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 33s 43ms/step - acc

(0.5085456371307373, 0.7966799736022949)

---

## Q6 — Model B: LSTM

Build an **LSTM-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add an **LSTM layer** to capture long-term dependencies in sequences.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [12]:
# ============================================================
# Question Q6 — Model B: LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build an LSTM-based sequence model
# 2) Add Input, Embedding, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential LSTM model
lstm_model = tf.keras.Sequential([
    
    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),
    
    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
    
    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS),
    
    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),
    
], name="lstm")

# TODO 6: Compile the model
lstm_model = compile_model(lstm_model, lr=0.001)

# Print model summary
lstm_model.summary()

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,411,713 (5.39 MB)

 Trainable params: 1,411,713 (5.39 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:

# ============================================================
# Train and Evaluate LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the LSTM model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the LSTM model
history_lstm = train_model(lstm_model, run_name="proj5_lstm", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(lstm_model, "Validation (LSTM)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(lstm_model, "Test (LSTM)", test_ds)


Epoch 1/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 93s 121ms/step - accuracy: 0.9472 - loss: 0.1497 - val_accuracy: 0.8640 - val_loss: 0.3629 - learning_rate: 6.2500e-05
Epoch 2/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 99s 130ms/step - accuracy: 0.9519 - loss: 0.1388 - val_accuracy: 0.8680 - val_loss: 0.3741 - learning_rate: 6.2500e-05
Epoch 3/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 102s 133ms/step - accuracy: 0.9570 - loss: 0.1284 - val_accuracy: 0.8660 - val_loss: 0.3734 - learning_rate: 3.1250e-05
Epoch 4/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 101s 132ms/step - accuracy: 0.9594 - loss: 0.1231 - val_accuracy: 0.8680 - val_loss: 0.3733 - learning_rate: 1.5625e-05
Epoch 5/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 96s 126ms/step - accuracy: 0.9611 - loss: 0.1203 - val_accuracy: 0.8680 - val_loss: 0.3750 - learning_rate: 7.8125e-06
Validation (LSTM): loss=0.3741, acc=0.8680
Test (LSTM): loss=0.3462, acc=0.8709


(0.3462429642677307, 0.8708800077438354)

---

## Q7 — Model C: GRU

Build a **GRU-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add **Embedding → GRU → Dense(sigmoid)** layers.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [15]:

# ============================================================
# Question Q7 — Model C: GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a GRU-based sequence model
# 2) Add Input, Embedding, GRU, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential GRU model
gru_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: GRU layer
    layers.GRU(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="gru")

# TODO 6: Compile the model
gru_model = compile_model(gru_model, lr=0.001)

# Print model summary
gru_model.summary()


Model: "gru"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,379,201 (5.26 MB)

 Trainable params: 1,379,201 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# ============================================================
# Train and Evaluate GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the GRU model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the GRU model
history_gru = train_model(gru_model, run_name="proj5_gru", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(gru_model, "Validation (GRU)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(gru_model, "Test (GRU)", test_ds)

Epoch 1/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 94s 120ms/step - accuracy: 0.7805 - loss: 0.4538 - val_accuracy: 0.7960 - val_loss: 0.4147 - learning_rate: 0.0010
Epoch 2/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 99s 130ms/step - accuracy: 0.9021 - loss: 0.2476 - val_accuracy: 0.8840 - val_loss: 0.2791 - learning_rate: 0.0010
Epoch 3/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 93s 121ms/step - accuracy: 0.9463 - loss: 0.1473 - val_accuracy: 0.8740 - val_loss: 0.3085 - learning_rate: 0.0010
Epoch 4/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 98s 128ms/step - accuracy: 0.9769 - loss: 0.0694 - val_accuracy: 0.8920 - val_loss: 0.3978 - learning_rate: 5.0000e-04
Epoch 5/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 90s 117ms/step - accuracy: 0.9911 - loss: 0.0331 - val_accuracy: 0.8820 - val_loss: 0.4592 - learning_rate: 2.5000e-04
Epoch 6/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 101s 132ms/step - accuracy: 0.9964 - loss: 0.0172 - val_accuracy: 0.8880 - val_loss: 0.4971 - learning_rate: 1.2500e-04
Epoch 7/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 99s 129ms/step - accurac

(0.41361406445503235, 0.8833600282669067)

---

## 8) Complex Model D — Stacked LSTM with Dropout

In this question, build a **deeper LSTM-based sequence classifier** using two recurrent layers.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `Dropout(...)`
  - `LSTM(...)`
  - `Dense(1, activation="sigmoid")`
- Train the model using the same optimizer and callbacks.
- Evaluate on validation and test sets.
- Compare it with the single-layer LSTM from Q6.

### Goal
Study whether **stacking recurrent layers** helps the model learn richer sequential sentiment patterns.


---

In [17]:
# ============================================================
# Question Q8 — Complex Model D: Stacked LSTM + Dropout (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a stacked LSTM model with dropout
# 2) Add Input, Embedding, LSTM, Dropout, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential stacked LSTM model
stacked_lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: First LSTM layer
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: Dropout layer
    layers.Dropout(.5),

    # TODO 6: Second LSTM layer
    layers.LSTM(RNN_UNITS // 2),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="stacked_lstm_dropout")

# TODO 8: Compile the model
stacked_lstm_model = compile_model(stacked_lstm_model, lr=0.001)

# Print model summary
stacked_lstm_model.summary()

# TODO 9: Train the model
history_stacked_lstm = train_model(
    stacked_lstm_model,
    run_name="proj5_stacked_lstm_dropout",
    epochs=8
)

# TODO 10: Evaluate on validation set
evaluate_model(stacked_lstm_model, "Validation (Stacked LSTM + Dropout)", val_ds)

# TODO 11: Evaluate on test set
evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)


Model: "stacked_lstm_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 300, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 300, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,461,057 (5.57 MB)

 Trainable params: 1,461,057 (5.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 166s 210ms/step - accuracy: 0.7811 - loss: 0.4681 - val_accuracy: 0.8300 - val_loss: 0.4054 - learning_rate: 0.0010
Epoch 2/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 165s 215ms/step - accuracy: 0.8667 - loss: 0.3248 - val_accuracy: 0.8340 - val_loss: 0.3895 - learning_rate: 0.0010
Epoch 3/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 166s 216ms/step - accuracy: 0.9057 - loss: 0.2389 - val_accuracy: 0.8520 - val_loss: 0.3376 - learning_rate: 0.0010
Epoch 4/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 164s 214ms/step - accuracy: 0.9327 - loss: 0.1771 - val_accuracy: 0.8420 - val_loss: 0.3854 - learning_rate: 0.0010
Epoch 5/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 155s 202ms/step - accuracy: 0.9586 - loss: 0.1158 - val_accuracy: 0.8560 - val_loss: 0.4424 - learning_rate: 5.0000e-04
Epoch 6/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 165s 215ms/step - accuracy: 0.9754 - loss: 0.0771 - val_accuracy: 0.8480 - val_loss: 0.4922 - learning_rate: 2.5000e-04
Epoch 7/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 167s 218ms/step - accur

(0.4578627347946167, 0.8694800138473511)

---

## 9) Complex Model E — Encoder–Decoder LSTM Classifier

In this question, build an **encoder–decoder style recurrent model** for sentiment classification.

### Idea
- The **encoder LSTM** reads the review and produces a compact context representation.
- A `RepeatVector` creates a short decoded sequence from that context.
- A **decoder LSTM** transforms the context into a richer hidden representation.
- A final dense layer predicts the review sentiment.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM` encoder
  - `RepeatVector(1)`
  - `LSTM` decoder
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it with the simpler one-layer LSTM.

### Goal
Explore whether a **more structured encoder–decoder design** is useful for sequence classification, even though it is more common in seq2seq tasks.


---

In [24]:

# ============================================================
# Question Q9 — Complex Model E: Encoder-Decoder LSTM Classifier (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define the input layer
# 2) Add Embedding, Encoder LSTM, RepeatVector, Decoder LSTM, and Dense layers
# 3) Build the functional model
# 4) Compile the model using the training utility
# 5) Train the model
# 6) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Define input layer
encdec_inputs = layers.Input(shape=(MAX_LEN,))

# TODO 2: Embedding layer
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(encdec_inputs)

# TODO 3: Encoder LSTM
encoded = layers.LSTM(RNN_UNITS)(x)

# TODO 4: Repeat encoded representation
decoded = layers.RepeatVector(MAX_LEN)(encoded)

# TODO 5: Decoder LSTM
decoded = layers.LSTM(RNN_UNITS // 2)(decoded)

# TODO 6: Output layer
encdec_outputs = layers.Dense(1, activation="sigmoid")(decoded)

# TODO 7: Build functional model
encdec_model = tf.keras.Model(encdec_inputs, encdec_outputs, name="encdec_lstm_classifier")


# TODO 8: Compile the model
encdec_model = compile_model(encdec_model, lr=0.001)

# TODO 9: Print model summary
encdec_model.summary()

# TODO 10: Train the model
history_encdec = train_model(
    encdec_model,
    run_name="proj5_encdec_lstm",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(encdec_model, "Validation (Encoder-Decoder LSTM)", val_ds) # i put history_encdec as the first arguement and had to retrain

# TODO 12: Evaluate on test set
evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)


Model: "encdec_lstm_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_12 (Embedding)        │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_14 (LSTM)                  │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_4 (RepeatVector)  │ (None, 300, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_15 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,461,057 (5.57 MB)

 Trainable params: 1,461,057 (5.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 157s 199ms/step - accuracy: 0.7925 - loss: 0.4504 - val_accuracy: 0.8580 - val_loss: 0.3517 - learning_rate: 0.0010
Epoch 2/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 156s 204ms/step - accuracy: 0.7802 - loss: 0.4590 - val_accuracy: 0.8420 - val_loss: 0.3600 - learning_rate: 0.0010
Epoch 3/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 148s 193ms/step - accuracy: 0.8764 - loss: 0.3088 - val_accuracy: 0.8600 - val_loss: 0.3503 - learning_rate: 5.0000e-04
Epoch 4/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 145s 189ms/step - accuracy: 0.9267 - loss: 0.1984 - val_accuracy: 0.8600 - val_loss: 0.3824 - learning_rate: 5.0000e-04
Epoch 5/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 151s 197ms/step - accuracy: 0.9502 - loss: 0.1445 - val_accuracy: 0.8560 - val_loss: 0.3912 - learning_rate: 2.5000e-04
Epoch 6/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 150s 196ms/step - accuracy: 0.9614 - loss: 0.1153 - val_accuracy: 0.8540 - val_loss: 0.4422 - learning_rate: 1.2500e-04
Validation (Encoder-Decoder LSTM): loss=0.3503, acc=0.

(0.3398483693599701, 0.8667200207710266)

---

## 10) Complex Model F — LSTM + GRU Hybrid

In this question, build a **hybrid recurrent architecture** that combines LSTM and GRU layers without using bidirectional processing.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `GRU(...)`
  - `Dropout`
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it against all earlier models in terms of accuracy and complexity.

### Goal
Test whether combining **LSTM-based memory** with a **GRU-based final sequence encoder** captures richer sentiment patterns than a single recurrent layer.


---

In [25]:
# ============================================================
# Question Q10 — Complex Model F: LSTM + GRU Hybrid (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a hybrid sequence model using LSTM and GRU layers
# 2) Add Input, Embedding, LSTM, GRU, Dropout, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential hybrid model
hybrid_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: GRU layer
    layers.GRU(RNN_UNITS // 2, dropout=0.2, recurrent_dropout=0.2),


    # TODO 6: Dropout layer
    layers.Dropout(0.5),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm_gru_hybrid")

# TODO 8: Compile the model
hybrid_model = compile_model(hybrid_model, lr=0.001)

# TODO 9: Print model summary
hybrid_model.summary()

# TODO 10: Train the model
history_hybrid = train_model(
    hybrid_model,
    run_name="proj5_lstm_gru_hybrid",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(hybrid_model, "Validation (LSTM + GRU Hybrid)", val_ds) # i put history_hybrid as the first arguement and had to retrain

# TODO 12: Evaluate on test set
evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)


Model: "lstm_gru_hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_13 (Embedding)        │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_16 (LSTM)                  │ (None, 300, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,448,897 (5.53 MB)

 Trainable params: 1,448,897 (5.53 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 212s 268ms/step - accuracy: 0.7683 - loss: 0.4836 - val_accuracy: 0.8040 - val_loss: 0.4595 - learning_rate: 0.0010
Epoch 2/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 209s 273ms/step - accuracy: 0.8700 - loss: 0.3210 - val_accuracy: 0.8440 - val_loss: 0.3819 - learning_rate: 0.0010
Epoch 3/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 219s 286ms/step - accuracy: 0.8886 - loss: 0.2857 - val_accuracy: 0.8300 - val_loss: 0.3814 - learning_rate: 0.0010
Epoch 4/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 220s 287ms/step - accuracy: 0.9287 - loss: 0.1942 - val_accuracy: 0.8580 - val_loss: 0.3684 - learning_rate: 0.0010
Epoch 5/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 234s 306ms/step - accuracy: 0.9446 - loss: 0.1555 - val_accuracy: 0.8640 - val_loss: 0.4524 - learning_rate: 0.0010
Epoch 6/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 262s 342ms/step - accuracy: 0.9660 - loss: 0.0990 - val_accuracy: 0.8600 - val_loss: 0.5318 - learning_rate: 5.0000e-04
Epoch 7/8
766/766 ━━━━━━━━━━━━━━━━━━━━ 276s 360ms/step - accuracy:

(0.5826364755630493, 0.8677999973297119)

---

## Q11 — Performance Comparison Table

Create a compact table comparing all completed models:

- Vanilla RNN
- LSTM
- GRU
- Stacked LSTM + Dropout
- Encoder–Decoder LSTM
- LSTM + GRU Hybrid

### Student Tasks
- Create a comparison table summarizing the results of all models.
- Include validation accuracy and test accuracy.
- Identify the best-performing recurrent model.
- Briefly comment on whether more complex architectures were helpful.


---

In [26]:

# ============================================================
# Question Q11 — Performance Comparison Table (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define helper functions for validation and training accuracy
# 2) Evaluate all trained models on the test dataset
# 3) Store model names and metrics in summary rows
# 4) Create a pandas DataFrame for comparison
# 5) Sort the table by test accuracy
# ============================================================

import pandas as pd

# TODO 1: Best validation accuracy from training history
def best_val_acc(history):
    return max(history.history.get("val_accuracy", [np.nan]))

# TODO 2: Final training accuracy from training history
def final_train_acc(history):
    return history.history.get("accuracy", [np.nan])[-1]


summary_rows = []
#pass the mode intot he eval model function and return the test accuacy
# TODO 3: Evaluate all models on the test set
_, rnn_test_acc = evaluate_model(rnn_model, "Test (Vanilla RNN)", test_ds)
_, lstm_test_acc = evaluate_model(lstm_model, "Test (LSTM)", test_ds)
_, gru_test_acc = evaluate_model(gru_model, "Test (GRU)", test_ds)
_, stacked_lstm_test_acc = evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)
_, encdec_test_acc = evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)
_, hybrid_test_acc = evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)
#get the train, val and test acc from the eval function and append the roles
# TODO 4: Append summary rows
summary_rows.append(["Vanilla RNN", final_train_acc(history_rnn), best_val_acc(history_rnn), rnn_test_acc])
summary_rows.append(["LSTM", final_train_acc(history_lstm), best_val_acc(history_lstm), lstm_test_acc])
summary_rows.append(["GRU", final_train_acc(history_gru), best_val_acc(history_gru), gru_test_acc])
summary_rows.append(["Stacked LSTM + Dropout", final_train_acc(history_stacked_lstm), best_val_acc(history_stacked_lstm), stacked_lstm_test_acc])
summary_rows.append(["Encoder-Decoder LSTM", final_train_acc(history_encdec), best_val_acc(history_encdec), encdec_test_acc])
summary_rows.append(["LSTM + GRU Hybrid", final_train_acc(history_hybrid), best_val_acc(history_hybrid), hybrid_test_acc])

# TODO 5: Create DataFrame
results_df = pd.DataFrame(
    summary_rows,
    columns=["Model", "Final Train Acc", "Best Val Acc", "Test Acc"]
)


# TODO 6: Sort by test accuracy
results_df = results_df.sort_values(by="Test Acc", ascending=False).reset_index(drop=True)


# Display results
results_df



Test (Vanilla RNN): loss=0.5085, acc=0.7967
Test (LSTM): loss=0.3462, acc=0.8709
Test (GRU): loss=0.4136, acc=0.8834
Test (Stacked LSTM + Dropout): loss=0.4579, acc=0.8695
Test (Encoder-Decoder LSTM): loss=0.3398, acc=0.8667
Test (LSTM + GRU Hybrid): loss=0.5826, acc=0.8678


,Model,Final Train Acc,Best Val Acc,Test Acc
0,GRU,0.997959,0.892,0.88336
1,LSTM,0.961102,0.868,0.87088
2,Stacked LSTM + Dropout,0.988490,0.860,0.86948
3,LSTM + GRU Hybrid,0.990898,0.872,0.86780
4,Encoder-Decoder LSTM,0.961388,0.860,0.86672
5,Vanilla RNN,0.914653,0.774,0.79668


---

# Results and Discussions

Use the results above to answer the discussion questions below. You may revise the answers based on your actual experimental results (**Q1-Q11**).

---

## **Q12** — Which model performed best overall, and why might it outperform the others?  

**Answer:** ...
From my data it looks like GRU performed the best. It had the highest Training accuracy, validation accuracy and test accuary. Some version of LSTM/GRU performed well. I think its because the dataset was suited better to LSTM/GRU type models. GRU being a more light weigght verion of LSTM makes sense. 
---

## **Q13** — Why does a vanilla RNN usually struggle more on long reviews?  

**Answer:** ...
I think vanilla RNN struggling with vanishing or expoding gradients while the newer RNN like LSTM and GRU have solutions for this
---

## **Q14** — Why might a more complex model not always outperform a simpler GRU or LSTM?  

**Answer:** ...
Some complex models used on a simple datasets can overfit and not learn properly. I think its best if you use the model designed for the data. That way you learn the data and not memorize it or overfit it
---

## **Q15** — What is the main idea behind the encoder–decoder classifier used here?  

**Answer:** ...
I think the dataset is positive review or negative review due to the bin classification. I dont think encoder/decoder performs on this type of data. I think econder/decoder would be better used in language translations like english to french example in the class. I think the main idea of it being here was to enforce this concept. 
---

### 🎉 Congratulations!

You have successfully completed **P5 — Sequence Modeling with RNNs**. Excellent work applying and comparing **RNN, LSTM, and GRU architectures** for a **sequence-based text classification** task using the **IMDB Movie Reviews dataset**.

### **Submission Instructions**

Please submit a **GitHub repository link** on Canvas that contains:
- The **completed Jupyter notebook**
- Notebook runs **top-to-bottom** without errors

Before submitting, ensure that:
- All **code cells (Q1–Q11)** have been executed successfully
- All **Markdown responses (Q12–Q15)** have been completed
- The notebook is **saved after execution** so that outputs are visible

Once verified, **push the final version to GitHub** and submit the repository link on Canvas.